In [246]:
using LowLevelFEM

In [247]:
structured_rect_mesh()
#openPreProcessor()

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:fu)
Pv = Problem([mat], type=:VectorField, dim=2, field=:v, rhs_field=:fv)
PT = Problem([mat], type=:ScalarField, dim=2, field=:T, rhs_field=:q)

Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q)

In [248]:
E = mat.E
ν = mat.ν
k = mat.k
α = mat.α
κ = mat.κ
ρ = mat.ρ
c = mat.c

T0 = 293

D = E / (1 - ν^2) * [1 ν 0; ν 1 0; 0 0 (1-ν)/2]

3×3 Matrix{Float64}:
     2.1978e5  65934.1           0.0
 65934.1           2.1978e5      0.0
     0.0           0.0       76923.1

In [249]:
Mu = ∫(Pu ⋅ Pu)
Mv = ∫(Pv ⋅ Pv * ρ)
MT = ∫(PT ⋅ PT * (c * ρ))

sparse([1, 5, 40, 41, 2, 13, 14, 113, 3, 22  …  121, 3, 21, 22, 23, 24, 111, 112, 120, 121], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  120, 121, 121, 121, 121, 121, 121, 121, 121, 121], [0.0036633333333333314, 0.0018316666666666663, 0.0018316666666666663, 0.0009158333333333332, 0.0036633333333333322, 0.0018316666666666666, 0.0018316666666666666, 0.0009158333333333332, 0.0036633333333333305, 0.0018316666666666663  …  0.003663333333333332, 0.0009158333333333332, 0.0009158333333333336, 0.0036633333333333327, 0.0036633333333333327, 0.0009158333333333332, 0.0009158333333333325, 0.0036633333333333322, 0.003663333333333332, 0.014653333333333326], 121, 121)

In [250]:
Muv = ∫(Pu ⋅ Pv * 0)
MuT = ∫(Pu ⋅ [0; 0;;] ⋅ PT)
Mvu = ∫(Pv ⋅ Pu * 0)
MvT = ∫(Pv ⋅ [0; 0;;] ⋅ PT)
MTu = ∫(PT ⋅ [0 0] ⋅ Pu)
MTv = ∫(PT ⋅ [0 0] ⋅ Pv)

sparse(Int64[], Int64[], Float64[], 121, 242)

In [251]:
αI = [1; 1; 0;;] * α

3×1 Matrix{Float64}:
 1.2e-5
 1.2e-5
 0.0

In [252]:
Kuv = ∫(Pu ⋅ Pv)

Kvu = ∫(SymGrad(Pv) ⋅ D ⋅ SymGrad(Pu))
KvT = ∫(SymGrad(Pv) ⋅ D ⋅ αI ⋅ PT)
KTu = ∫(PT ⋅ Div(Pu) * (κ * T0))

KTT = ∫(Grad(PT) ⋅ k ⋅ Grad(PT))

sparse([1, 5, 40, 41, 2, 13, 14, 113, 3, 22  …  121, 3, 21, 22, 23, 24, 111, 112, 120, 121], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  120, 121, 121, 121, 121, 121, 121, 121, 121, 121], [29.999999999999993, -7.499999999999996, -7.500000000000001, -15.0, 29.999999999999996, -7.500000000000001, -7.499999999999997, -15.0, 30.0, -7.500000000000001  …  -15.00000000000001, -15.000000000000004, -15.00000000000001, -14.999999999999993, -15.0, -14.999999999999998, -14.999999999999993, -14.99999999999999, -15.00000000000001, 120.0], 121, 121)

In [253]:
Kuu = ∫(Pu ⋅ Pu * 0)
KuT = ∫(Pu ⋅ [0; 0;;] ⋅ PT)

Kvv = ∫(Pv ⋅ Pv * 0)

KTv = ∫(PT ⋅ [0 0] ⋅ Pv)

sparse(Int64[], Int64[], Float64[], 121, 242)

In [254]:
K = SystemMatrix([
    Kuu -Kuv KuT;
    Kvu Kvv KvT;
    KTu KTv KTT
])

K[:, :]

605×605 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10358 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡻⣌⠐⠒⠂⠃⠶⠰⠠⠀⠄⡄⡄⢀⢈⠐⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠇⠈⠳⣄⠀⠀⠀⢀⢀⢀⠀⡀⡄⡄⠨⠳⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣭⠀⠀⠈⢳⣴⣧⡈⠈⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢒⡀⠀⢐⡋⠺⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⠄⠀⠐⠀⠀⠈⠻⣿⣟⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠅⠀⠭⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣒⠀⡉⠀⠀⠀⠀⠀⠀⠈⠻⣿⣵⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡂⠰⣔⡂⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣟⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡛⢮⡉⠁⠃⠛⠘⠰⠀⠆⠆⠄⢠⢠⢈⡈⠁⠀⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢘⢯⠙⠛⠳⠶⠄⣤⡉⎥
⎢⢰⠀⠹⢦⡀⠀⢀⢀⢀⠀⡄⡄⡄⠠⠰⠹⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡎⢧⠀⣀⣠⣤⠠⠿⎥
⎢⠬⠀⠀⠀⢙⣶⣫⣈⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⠅⠈⣿⡍⠀⠀⠀⠀⎥
⎢⢘⡃⠀⢀⡉⠻⣿⣿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡃⢈⢹⣿⡄⠀⠀⠀⎥
⎢⠀⠂⠀⢐⠂⠀⠈⠻⣿⢿⣶⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠂⢐⠀⢻⣿⡆⠀⠀⎥
⎢⠀⠥⠀⠠⠀⠀⠀⠀⠀⠻⣿⣿⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠥⠠⠀⠀⢻⣿⣄⠀⎥
⎢⠀⢉⠀⠭⠀⠀⠀⠀⠀⠀⠈⠻⢟⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢉⠭⠀⠀⠀⠻⣿⡄⎥
⎢⣂⢐⣦⡂⠀⣀⠀⠀⠀⠀⠀⠀⠈⠻⣿⢿⡃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣰⡆⣀⠀⠀⠀⢻⣿⎥
⎢⢫⠛⠾⣥⣁⠉⣈⢈⢀⠁⡁⡃⡜⢰⠰⠮⠅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡿⣯⣉⣉⣉⣣⢳⠮⎥
⎢⢻⡄⠀⢀⡿⠿⣷⣶⣤⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⢸⢻⣶⣄⠀⠀⠀⎥
⎢⠀⠧⠀⣸⠁⠀⠀⠉⠛⠿⣿⣶⣤⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠧⣸⠀⠙⢿⣷⣄⠀⎥
⎣⢄⠹⣤⡖⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣿⣶⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⡹⡖⠀⠀⠀⠙⢿⣷⎦

In [255]:
fv = ∫(Pv ⋅ [1.0, 0.0], Γ="right")
fT = ∫(PT ⋅ 1.0, Γ="right")

nodal ScalarField
[0.0; 0.05; … ; 0.0; 0.0;;]

In [256]:
T0 = 0
fv_T0 = ∫(SymGrad(Pv) ⋅ D ⋅ αI ⋅ T0)

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [257]:
fu = ∫(Pu ⋅ [0, 0])

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [258]:
F = SystemVector([fu, fv + fv_T0, fT])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :fu), Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q)], [0, 242, 484])

In [259]:
M = SystemMatrix([Mu Muv MuT; Mvu Mv MvT; MTu MTv MT])

M = consistentToLumped(M)

M[:, :]

605×605 SparseArrays.SparseMatrixCSC{Float64, Int64} with 605 stored entries:
⎡⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⎦

In [260]:
supp1 = BoundaryCondition("left", problem=Pu, ux=0)
supp2 = BoundaryCondition("bottom", problem=Pu, uy=0)

supp3 = BoundaryCondition("left", problem=Pv, vx=0)
supp4 = BoundaryCondition("bottom", problem=Pv, vy=0)

suppT = BoundaryCondition("left", problem=PT, T=0)

BoundaryCondition("left", Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q), Dict{Symbol, Union{Function, Number, ScalarField}}(:T => 0))

In [261]:
X0 = SystemVector([0fu, 0fv, 0fT])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :fu), Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q)], [0, 242, 484])

In [262]:
KK = SystemMatrix([Kuu Kuv; Kvu Kvv])
MM = SystemMatrix([Mu Muv; Mvu Mv])

mu, mv = solveEigenFields(KK, MM)
λmax = maximum(mu.f)
λmax = largestEigenValue(KK, MM)
dt = 2 / √(λmax)

┌ Warning: largestEigenValue: relative residual too large: 0.7318322507638586
└ @ LowLevelFEM ~/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl:3301


0.00013655113712936505

In [263]:
λmax = largestEigenValue(KTT, MT)
dt = 2 / λmax

6.105555555555543e-5

In [264]:
u, v, T = FDM(K, M, F, X0, 100, 0.001, support=[supp1, supp2, supp3, supp4, suppT], ϑ=0.6)

(VectorField(Matrix{Float64}[], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 8.082705477228481e-6 … 7.979719256098974e7 1.1057527668519974e8; 0.0 -1.553725303904993e-6 … 1.595594600517943e8 2.211021589521071e8], [0.0, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009000000000000001  …  0.09000000000000007, 0.09100000000000007, 0.09200000000000007, 0.09300000000000007, 0.09400000000000007, 0.09500000000000007, 0.09600000000000007, 0.09700000000000007, 0.09800000000000007, 0.09900000000000007], Int64[], 100, :v2D, Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :fu)), VectorField(Matrix{Float64}[], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.013239045920144603 … 2.4601021838458298e10 3.4089730676423775e10; 0.0 -0.00254781171289863

## Stabilitás-ellenőrzés

Ez a cella kiírja minden eltárolt időpillanatra a `max|u|`, `max|v|`, `max|T|` értékeket. Ha ezek lépésről lépésre sok nagyságrendet ugranak, akkor a megoldás numerikusan elszáll.


In [265]:
function field_matrix_for_diagnostics(f)
    for name in propertynames(f)
        val = getproperty(f, name)
        if val isa AbstractMatrix{<:Real}
            return val
        end
    end
    error("Nem találtam numerikus mátrixot ebben a mezőben. propertynames = $(propertynames(f))")
end

function max_per_time_step(f)
    A = field_matrix_for_diagnostics(f)
    return [maximum(abs.(A[:, j])) for j in axes(A, 2)]
end

max_u = max_per_time_step(u)
max_v = max_per_time_step(v)
max_T = max_per_time_step(T)

println("step | max|u| | max|v| | max|T|")
println("----------------------------------")
for j in eachindex(max_u)
    println(j - 1, " | ", max_u[j], " | ", max_v[j], " | ", max_T[j])
end

println()
println("Utolsó értékek:")
println("max|u| = ", max_u[end])
println("max|v| = ", max_v[end])
println("max|T| = ", max_T[end])

step | max|u| | max|v| | max|T|
----------------------------------
0 | 0.0 | 0.0 | 0.0
1 | 9.047945573375638e-6 | 0.015954626631424954 | 0.06561274649532363
2 | 4.464053822870918e-6 | 0.018913398100818896 | 0.15963593675412494
3 | 9.504297279802657e-6 | 0.021382376539083554 | 0.2943438814186536
4 | 8.880102763528606e-6 | 0.015887164417121614 | 0.4881546856795409
5 | 1.3077506766030044e-5 | 0.017680472427233283 | 0.7654494903811332
6 | 1.5511796658672022e-5 | 0.009133215147735891 | 1.1599962490946156
7 | 2.1135091035650975e-5 | 0.015003072441867419 | 1.7157311281333505
8 | 2.8022277723622695e-5 | 0.013232511889210543 | 2.494961678233048
9 | 4.06350727210592e-5 | 0.01659533983976751 | 3.583864345026606
10 | 5.8399094986018015e-5 | 0.022881100979822436 | 5.101719126707019
11 | 8.297022445112078e-5 | 0.02941006851850581 | 7.213688302818361
12 | 0.00011719925975384354 | 0.042628808613571344 | 10.148512863590494
13 | 0.0001646660492650817 | 0.057909214097034235 | 14.223025601563505
14 | 0.00

In [266]:
showDoFResults(u, name="u", factor=0.000001, visible=true)
showDoFResults(v, name="v", factor=0.000001)
showDoFResults(T, name="T")

2

In [267]:
showDoFResults(T, name="T")

3

In [268]:
εx = ∂x(u[1])
εy = ∂y(u[2])
γxy = ∂y(u[1]) + ∂x(u[2])

ε = [εx, εy, γxy]

3-element Vector{ScalarField}:
 ScalarField([[0.0 8.507018306270975e-6 … 2.0880575930344287e7 2.893429450452917e7; 0.0 8.507018306270977e-6 … 2.0880575930344287e7 2.8934294504529174e7; 0.0 8.507229961853882e-6 … 2.0792530176450692e7 2.881228916568382e7; 0.0 8.507229961853884e-6 … 2.0792530176450692e7 2.881228916568382e7], [0.0 8.507229961853884e-6 … 2.0792530176450692e7 2.881228916568382e7; 0.0 8.507229961853884e-6 … 2.0792530176450692e7 2.881228916568382e7; 0.0 8.507879788202128e-6 … 2.0485347785152085e7 2.838662534664884e7; 0.0 8.507879788202128e-6 … 2.0485347785152085e7 2.838662534664884e7], [0.0 8.507879788202128e-6 … 2.0485347785152085e7 2.838662534664884e7; 0.0 8.50787978820213e-6 … 2.0485347785152085e7 2.838662534664884e7; 0.0 8.509005437277266e-6 … 1.9821303165704343e7 2.746645616405264e7; 0.0 8.509005437277266e-6 … 1.9821303165704347e7 2.746645616405264e7], [0.0 8.509005437277266e-6 … 1.9821303165704347e7 2.746645616405264e7; 0.0 8.509005437277266e-6 … 1.9821303165704347e7 2.7

In [269]:
T = nodesToElements(T)
εth = [α * T, α * T, 0]

3-element Vector{Any}:
  ScalarField([[0.0 0.0 … 0.0 0.0; 0.0 -4.836526487736796e-7 … -3.9953356668512926e7 -5.536352044454615e7; 0.0 -4.835592647893169e-7 … -3.9780740554408714e7 -5.512432562226975e7; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 -4.835592647893169e-7 … -3.9780740554408714e7 -5.512432562226975e7; 0.0 -4.832718270897541e-7 … -3.925840641919222e7 -5.440052517562728e7; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 -4.832718270897541e-7 … -3.925840641919222e7 -5.440052517562728e7; 0.0 -4.827671982688135e-7 … -3.837334881241033e7 -5.3174097403854825e7; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 -4.827671982688135e-7 … -3.837334881241033e7 -5.3174097403854825e7; 0.0 -4.820022295556214e-7 … -3.710559801198291e7 -5.141737018935184e7; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 -4.820022295556214e-7 … -3.710559801198291e7 -5.141737018935184e7; 0.0 -4.809057819331097e-7 … -3.5431720748115e7 -4.909787196406368e7; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 -4.809057819331097e-7 … -

In [270]:
σ = D * (ε - εth)

3-element Vector{ScalarField}:
 ScalarField([[0.0 1.7449041900360387 … 1.0057901482797852e13 1.3937272851868686e13; 0.0 1.8833665759185756 … 2.160810122095979e13 2.994242918182773e13; 0.0 1.8833864124907147 … 2.153943161607428e13 2.9847273443848684e13; 0.0 1.744950707746568 … 1.0038550767656402e13 1.3910458491682895e13], [0.0 1.7447213600645957 … 9.939962644833592e12 1.3773844548374512e13; 0.0 1.8831844127159008 … 2.144675672400209e13 2.97188534870047e13; 0.0 1.8832451066365188 … 2.1230005786182297e13 2.9418500871692566e13; 0.0 1.7448641790422317 … 9.872450031361371e12 1.3680292060674516e13], [0.0 1.7443817096703917 … 9.6744736566565e12 1.3405955432338934e13; 0.0 1.8828241566533068 … 2.1044488430575797e13 2.9161428753791574e13; 0.0 1.8829273723913094 … 2.064567095623202e13 2.860878583381347e13; 0.0 1.7446291050715208 … 9.528529784250404e12 1.3203720447152957e13], [0.0 1.7438406719401414 … 9.229941389138674e12 1.2789965353422176e13; 0.0 1.882251196555847 … 2.0367458407732547e13 2.822326

In [271]:
σx = σ[1]
σy = σ[2]
τxy = σ[3]

elementwise ScalarField
[[0.0 0.0 … 0.0 0.0; 0.0 1.6281198685450646e-5 … -6.77275029950885e9 -9.385026065029139e9; 0.0 0.0003381818037513487 … 1.506747213851116e11 2.08790544038472e11; 0.0 0.00032190060506587324 … 1.5744747168462094e11 2.1817557010350204e11], [0.0 0.00032190060506587324 … 1.5744747168462094e11 2.1817557010350204e11; 0.0 0.0003718872472384716 … 1.3381805696934348e11 1.854321994085034e11; 0.0 0.0007256937439888061 … 2.981642978630192e11 4.1316742038714056e11; 0.0 0.0006757071018162077 … 3.2179371257829663e11 4.459107910821392e11], [0.0 0.0006757071018162077 … 3.2179371257829663e11 4.459107910821392e11; 0.0 0.000762295492211232 … 2.707133572361702e11 3.7512854626705585e11; 0.0 0.0011878746090267906 … 4.4959512041127454e11 6.230056960860497e11; 0.0 0.001101286218631705 … 5.006754757534013e11 6.937879409011282e11], [0.0 0.001101286218631705 … 5.006754757534013e11 6.937879409011282e11; 0.0 0.0012265219094754142 … 4.0191840236223096e11 5.569398840732958e11; 0.0 0.001783067871

In [272]:
showElementResults(σx, name="σx")
showElementResults(σy, name="σy")
showElementResults(τxy, name="τxy")

6

In [273]:
openPostProcessor()